# Week 6 Coding Practice: Clustering as a Modeling Workflow

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/obscrivn/DataScience-book/blob/main/appliedDataScience/module06/week6_clustering_practice.ipynb)

**Estimated time:** 35–45 minutes  
**Format:** worked examples, interpretation checkpoints, and a short companion clustering comparison

This notebook is about more than fitting a clustering algorithm. We will compare how the same observations are grouped under different feature choices and preprocessing decisions, choose a sensible K, and interpret the cluster patterns we find.


## Learning objectives

By the end of this activity, you should be able to:

1. choose a feature set for a clustering task;
2. explain why scaling can change a clustering result;
3. fit and interpret K-means with multiple values of K;
4. use WSS/inertia and silhouette score as evidence rather than as automatic answers;
5. interpret clusters using the original variables; and
6. judge whether a clustering result is credible and stable enough to use.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.cluster.hierarchy import dendrogram, linkage
from sklearn.cluster import KMeans
from sklearn.datasets import make_blobs
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

sns.set_theme(style="whitegrid", context="notebook")

base, _ = make_blobs(
    n_samples=300,
    centers=[(-4, -5, 0, 0), (0, 0, 0, 0), (5, 5, 8, 8)],
    cluster_std=[1.0, 1.4, 1.1],
    random_state=42,
)
X_raw = np.column_stack([base[:, 0], base[:, 1] * 25, base[:, 2], base[:, 3] * 0.5])
feature_names = ["feature_1", "feature_2", "feature_3", "feature_4"]
df = pd.DataFrame(X_raw, columns=feature_names)
df.head()


## 1. Choose a clustering question and inspect the feature scales

We are not told the 'right' groups ahead of time. Instead, we must decide which features to use and then look for structure in the observations.

Before we fit a model, answer briefly:

- Which variable is likely to dominate Euclidean distance calculations if we do not scale the features?
- Why might scaling change the cluster assignments?
- What do we need to look at besides a numerical metric before deciding whether a clustering solution is useful?


In [ ]:
X_scaled = StandardScaler().fit_transform(X_raw)

raw_model = KMeans(n_clusters=3, n_init=20, random_state=42)
scaled_model = KMeans(n_clusters=3, n_init=20, random_state=42)

raw_labels = raw_model.fit_predict(X_raw)
scaled_labels = scaled_model.fit_predict(X_scaled)

print(f"Raw silhouette score: {silhouette_score(X_raw, raw_labels):.3f}")
print(f"Scaled silhouette score: {silhouette_score(X_scaled, scaled_labels):.3f}")
print("Raw cluster sizes:")
print(pd.Series(raw_labels).value_counts().sort_index().to_dict())
print("Scaled cluster sizes:")
print(pd.Series(scaled_labels).value_counts().sort_index().to_dict())


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharex=True, sharey=True)
sns.scatterplot(
    data=df.assign(cluster=raw_labels),
    x="feature_1", y="feature_2", hue="cluster", palette="Set2", s=40, ax=axes[0],
)
axes[0].set_title("Clusters on raw features")

scaled_df = df.copy()
scaled_df[["feature_1", "feature_2", "feature_3", "feature_4"]] = X_scaled
sns.scatterplot(
    data=scaled_df.assign(cluster=scaled_labels),
    x="feature_1", y="feature_2", hue="cluster", palette="Set2", s=40, ax=axes[1],
)
axes[1].set_title("Clusters after scaling")
plt.tight_layout()
plt.show()


### Interpretation checkpoint

What changed when we scaled the features? Write 2–3 sentences that explain the result in terms of distance, not just in terms of the plotted colors. Include a brief note about why one feature with a much larger range can dominate Euclidean distance.


In [ ]:
ks = range(2, 7)
inertia = []
silhouettes = []
for k in ks:
    model = KMeans(n_clusters=k, n_init=25, random_state=42)
    labels = model.fit_predict(X_scaled)
    inertia.append(model.inertia_)
    silhouettes.append(silhouette_score(X_scaled, labels))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(list(ks), inertia, marker='o')
axes[0].set_title("Elbow plot (WSS / inertia)")
axes[0].set_xlabel("K")
axes[0].set_ylabel("Inertia")
axes[1].plot(list(ks), silhouettes, marker='o', color='C1')
axes[1].set_title("Silhouette score by K")
axes[1].set_xlabel("K")
axes[1].set_ylabel("Silhouette score")
plt.tight_layout()
plt.show()
for k, sil in zip(ks, silhouettes):
    print(f"K = {k}: silhouette = {sil:.3f}")


## 2. Choose a reasonable K and justify it

There is rarely one objectively correct value of K. The right goal is to choose a value that balances fit, interpretability, and stability.

Use the elbow plot and silhouette scores as evidence. Then decide which K is most defensible for this task and write a short justification in 2–4 sentences.

- Which K would you choose and why?
- What evidence supports the choice?
- What evidence would make you hesitate?


In [ ]:
k_choice = 3
final_model = KMeans(n_clusters=k_choice, n_init=25, random_state=42)
df["cluster"] = final_model.fit_predict(X_scaled)
cluster_profile = df.groupby("cluster")[feature_names].mean().round(2)
cluster_profile


## 3. Interpret the clusters using original variables

Cluster labels like 0, 1, and 2 are just identifiers. They become meaningful only when we inspect the original variables and ask what separates one group from another.

Look at the cluster means above. Then answer:

- Which cluster has the largest average `feature_2`? Why might that matter?
- Are the clusters easy to describe in plain language?
- What does the clustering result not establish?


In [ ]:
alt_model = KMeans(n_clusters=4, n_init=25, random_state=42)
alt_labels = alt_model.fit_predict(X_scaled)
print("Silhouette for 4 clusters:", silhouette_score(X_scaled, alt_labels))
print(pd.Series(alt_labels).value_counts().sort_index().to_dict())


### Sensitivity checkpoint

A reasonable decision changed: we asked for 4 clusters instead of 3. Did the cluster structure stay similar? Write a short note explaining whether your substantive interpretation remains stable or whether it changes materially.


In [ ]:
Z = linkage(X_scaled, method="ward")
plt.figure(figsize=(10, 5))
dendrogram(Z, p=12, truncate_mode="lastp", no_labels=True)
plt.title("Hierarchical clustering dendrogram")
plt.xlabel("Observation index")
plt.ylabel("Ward distance")
plt.tight_layout()
plt.show()


## 4. A shorter look at hierarchical clustering

Hierarchical clustering builds a nested set of merges instead of fixing K up front. The dendrogram shows the order in which clusters are combined. If we cut the tree at a chosen height, we get a set of groups.

How does the dendrogram help you think about cluster structure compared with K-means? What would you want to inspect before treating a hierarchical split as meaningful?


## Final AI-era checkpoint

An AI assistant says:

> "The elbow plot proves that the dataset has three natural groups."

Evaluate this statement using the evidence from the notebook. What is overstated, and what extra evidence would you want before accepting the interpretation?


## Final reflection

Write a short conclusion that answers this question:

Why did these clusters emerge, how much do I trust them, and which modeling choices helped produce them?

Your answer should mention at least two of the following: feature selection, scaling, K, silhouette evidence, cluster profiles, and stability under a different modeling choice.
